In [1]:
from google.colab import drive
import os
import json
import glob
import numpy as np
import shutil

drive.mount('/content/drive')

MASTER_OUTPUT_DIR = "/content/drive/MyDrive/GENAI_TP2/outputs"

print(f"Drive mounted. {MASTER_OUTPUT_DIR}")

Mounted at /content/drive
Drive mounted. /content/drive/MyDrive/GENAI_TP2/outputs


In [ ]:

def print_phase_stats(phase_name, pool_data, target_metrics):
    print(f"- Phase: {phase_name}")
    for m in target_metrics:
        vals = pool_data[m]
        if vals:
            print(f"- {m:<8} -> Mean: {float(np.mean(vals)):.5f} | Std: {float(np.std(vals)):.5f}  (N={len(vals)})")
        
def evaluate_single_pipeline(pipeline_dir):
    pipeline_name = os.path.basename(pipeline_dir)
    print("=" * 80)
    print(f"[*] EVALUATING PIPELINE: {pipeline_name}")
    print(f"[*] Path: {pipeline_dir}")
    print("=" * 80)

    run_folders = sorted([f.path for f in os.scandir(pipeline_dir) if f.is_dir() and "TOP_3_RESULTS" not in f.name])

    target_metrics = ['fitness', 'clip', 'lpips', 'rmse']

    pools = {
        'vlm': {m: [] for m in target_metrics},
        'opro_pop': {m: [] for m in target_metrics},
        'opro_best': {m: [] for m in target_metrics},
        'ga_pop': {m: [] for m in target_metrics},
        'ga_best': {m: [] for m in target_metrics}
    }

    run_rankings = []

    valid_run_count = 0

    for run_path in run_folders:
        run_name = os.path.basename(run_path)
        opro_chk_dir = os.path.join(run_path, "OPRO", "checkpoints")
        ga_chk_dir = os.path.join(run_path, "GA", "checkpoints")
        opro_iter_1_path = os.path.join(opro_chk_dir, "opro_iter_001.json")

        valid_run = False

        if os.path.exists(opro_iter_1_path):
            with open(opro_iter_1_path, 'r') as f:
                opro_data = json.load(f)
            if isinstance(opro_data, list):
                vlm_entries = [e for e in opro_data if 'iteration' not in e]
                for entry in vlm_entries:
                    for m in target_metrics:
                        if m in entry and entry[m] is not None:
                            pools['vlm'][m].append(entry[m])
                valid_run = True

        opro_files = sorted(glob.glob(os.path.join(opro_chk_dir, "opro_iter_*.json")))
        if opro_files:
            with open(opro_files[-1], 'r') as f:
                opro_final_data = json.load(f)
            if isinstance(opro_final_data, list) and len(opro_final_data) > 0:
                best_opro = opro_final_data[0]
                for m in target_metrics:
                    if m in best_opro and best_opro[m] is not None:
                        pools['opro_best'][m].append(best_opro[m])

                for candidate in opro_final_data:
                    for m in target_metrics:
                        if m in candidate and candidate[m] is not None:
                            pools['opro_pop'][m].append(candidate[m])

        ga_files = sorted(glob.glob(os.path.join(ga_chk_dir, "ga_iter_*.json")))
        if ga_files:
            last_file_path = ga_files[-1]
            with open(last_file_path, 'r') as f:
                ga_final_data = json.load(f)

            if isinstance(ga_final_data, list) and len(ga_final_data) > 0:
                best_candidate = ga_final_data[0]

                for m in target_metrics:
                    if m in best_candidate and best_candidate[m] is not None:
                        pools['ga_best'][m].append(best_candidate[m])

                if 'fitness' in best_candidate and best_candidate['fitness'] is not None:
                    iter_num = os.path.basename(last_file_path).split('_')[-1].split('.')[0]
                    best_img_path = os.path.join(run_path, "GA", "images", f"best_iter_{iter_num}.png")

                    run_rankings.append({
                        'pipeline_name': pipeline_name,
                        'run_name': run_name,
                        'fitness': float(best_candidate['fitness']),
                        'clip': float(best_candidate.get('clip', 0.0)),
                        'lpips': float(best_candidate.get('lpips', 0.0)),
                        'rmse': float(best_candidate.get('rmse', 0.0)),
                        'prompt': best_candidate.get('prompt', 'No prompt string found in JSON.'),
                        'image_path': best_img_path
                    })

                for candidate in ga_final_data:
                    for m in target_metrics:
                        if m in candidate and candidate[m] is not None:
                            pools['ga_pop'][m].append(candidate[m])

        if valid_run:
            valid_run_count += 1

    print(f"PIPELINE METRICS (Total Runs: {valid_run_count})")
    print_phase_stats("VLM Baseline", pools['vlm'], target_metrics)
    print_phase_stats("Final OPRO (BEST Candidates Only)", pools['opro_best'], target_metrics)
    print_phase_stats("Final OPRO (Entire Population)", pools['opro_pop'], target_metrics)
    print_phase_stats("Final GA (BEST Candidates Only)", pools['ga_best'], target_metrics)
    print_phase_stats("Final GA (Entire Population)", pools['ga_pop'], target_metrics)
    print("\n")

    return run_rankings, pools, valid_run_count

def merge_pools(global_pool, local_pool, target_metrics):
    for category in global_pool.keys():
        for m in target_metrics:
            global_pool[category][m].extend(local_pool[category][m])

def evaluate_all_directories(master_dir):
    if not os.path.exists(master_dir):
        print(f"[!] ERROR: Master directory not found: {master_dir}")
        return

    pipeline_folders = sorted([f.path for f in os.scandir(master_dir) if f.is_dir() and "TOP_3_RESULTS" not in f.name])


    print(f"Found {len(pipeline_folders)} pipelines. Starting evaluation...\n")

    target_metrics = ['fitness', 'clip', 'lpips', 'rmse']

    global_inicial_pools = {c: {m: [] for m in target_metrics} for c in ['vlm', 'opro_pop', 'opro_best', 'ga_pop', 'ga_best']}
    global_nova_pools = {c: {m: [] for m in target_metrics} for c in ['vlm', 'opro_pop', 'opro_best', 'ga_pop', 'ga_best']}

    inicial_rankings_by_target = {}

    runs_inicial = 0
    runs_nova = 0

    for pipeline_dir in pipeline_folders:
        pipeline_name = os.path.basename(pipeline_dir)
        rankings, local_pools, run_count = evaluate_single_pipeline(pipeline_dir)

        target_name = pipeline_name.split('-')[0]

        if "INICIAL" in pipeline_name.upper():
            merge_pools(global_inicial_pools, local_pools, target_metrics)
            runs_inicial += run_count

            if target_name not in inicial_rankings_by_target:
                inicial_rankings_by_target[target_name] = []
            inicial_rankings_by_target[target_name].extend(rankings)

        elif "NOVA" in pipeline_name.upper():
            merge_pools(global_nova_pools, local_pools, target_metrics)
            runs_nova += run_count

    print("=" * 80)
    print(f"GLOBAL AVERAGES: PIPELINES 'INICIAL' (Total Runs: {runs_inicial})")
    print("=" * 80)
    print_phase_stats("VLM Baseline", global_inicial_pools['vlm'], target_metrics)
    print_phase_stats("Final OPRO (BEST Candidates Only)", global_inicial_pools['opro_best'], target_metrics)
    print_phase_stats("Final OPRO (Entire Population)", global_inicial_pools['opro_pop'], target_metrics)
    print_phase_stats("Final GA (BEST Candidates Only)", global_inicial_pools['ga_best'], target_metrics)
    print_phase_stats("Final GA (Entire Population)", global_inicial_pools['ga_pop'], target_metrics)
    print("\n")

    print("=" * 80)
    print(f"GLOBAL AVERAGES: PIPELINES 'NOVA' (Total Runs: {runs_nova})")
    print("=" * 80)
    print_phase_stats("VLM Baseline", global_nova_pools['vlm'], target_metrics)
    print_phase_stats("Final OPRO (BEST Candidates Only)", global_nova_pools['opro_best'], target_metrics)
    print_phase_stats("Final OPRO (Entire Population)", global_nova_pools['opro_pop'], target_metrics)
    print_phase_stats("Final GA (BEST Candidates Only)", global_nova_pools['ga_best'], target_metrics)
    print_phase_stats("Final GA (Entire Population)", global_nova_pools['ga_pop'], target_metrics)
    print("\n")

    print("=" * 80)
    print("TOP 3 EXTRACTION (PIPELINES INICIAL ONLY)")
    print("=" * 80)

    export_dir = os.path.join(master_dir, "GLOBAL_TOP_3_RESULTS")
    os.makedirs(export_dir, exist_ok=True)

    for target_name, rankings in inicial_rankings_by_target.items():
        if not rankings:
            continue

        rankings.sort(key=lambda x: x['fitness'], reverse=True)
        top_runs = rankings[:3]

        print(f"TOP 3 FOR TARGET: {target_name}")

        for i, rank_data in enumerate(top_runs):
            rank_num = i + 1
            placement = "1st" if i == 0 else "2nd" if i == 1 else "3rd"

            print(f" [{placement}] Pipeline: {rank_data['pipeline_name']} | Run: {rank_data['run_name']}")
            print(f"    -> Fitness: {rank_data['fitness']:.5f} | CLIP: {rank_data['clip']:.5f} | LPIPS: {rank_data['lpips']:.5f} | RMSE: {rank_data['rmse']:.5f}")

            base_filename = f"{target_name}_Rank_{rank_num}_Fitness_{rank_data['fitness']:.3f}"
            prompt_filename = os.path.join(export_dir, f"{base_filename}_prompt.txt")

            with open(prompt_filename, "w") as pf:
                pf.write(f"Target Image: {target_name}\n")
                pf.write(f"Source Pipeline: {rank_data['pipeline_name']}\n")
                pf.write(f"Source Run: {rank_data['run_name']}\n")
                pf.write(f"Final Fitness: {rank_data['fitness']:.5f}\n")
                pf.write(f"Final CLIP: {rank_data['clip']:.5f}\n")
                pf.write(f"Final LPIPS: {rank_data['lpips']:.5f}\n")
                pf.write(f"Final RMSE: {rank_data['rmse']:.5f}\n")
                pf.write(f"Prompt:{rank_data['prompt']}\n")

            src_img = rank_data['image_path']
            if os.path.exists(src_img):
                img_ext = os.path.splitext(src_img)[1]
                dst_img = os.path.join(export_dir, f"{base_filename}_image{img_ext}")
                shutil.copy2(src_img, dst_img)


evaluate_all_directories(MASTER_OUTPUT_DIR)

[*] Found 12 pipelines. Starting evaluation...

[*] EVALUATING PIPELINE: ASTRONAUTA-PIPELINE-INICIAL-50GA
[*] Path: /content/drive/MyDrive/GENAI_TP2/outputs/ASTRONAUTA-PIPELINE-INICIAL-50GA

[ PIPELINE METRICS (Total Runs: 5) ]

  - Phase: VLM Baseline
    - fitness  -> Mean: 0.74995 | Std: 0.02206  (N=50)
    - clip     -> Mean: 0.84042 | Std: 0.04065  (N=50)
    - lpips    -> Mean: 0.53668 | Std: 0.04361  (N=50)
    - rmse     -> Mean: 0.13465 | Std: 0.01323  (N=50)

  - Phase: Final OPRO (BEST Candidates Only)
    - fitness  -> Mean: 0.79267 | Std: 0.00876  (N=5)
    - clip     -> Mean: 0.88611 | Std: 0.00910  (N=5)
    - lpips    -> Mean: 0.43671 | Std: 0.04331  (N=5)
    - rmse     -> Mean: 0.11259 | Std: 0.00513  (N=5)

  - Phase: Final OPRO (Entire Population)
    - fitness  -> Mean: 0.76812 | Std: 0.01162  (N=100)
    - clip     -> Mean: 0.86807 | Std: 0.02218  (N=100)
    - lpips    -> Mean: 0.50599 | Std: 0.04022  (N=100)
    - rmse     -> Mean: 0.12825 | Std: 0.01201  (N=100